In [1]:
# %% [1] ИМПОРТ БИБЛИОТЕК И НАСТРОЙКА

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, roc_curve

# Настройка визуализации
sns.set_theme(style="whitegrid")

# %% [2] ЗАГРУЗКА И ПЕРВИЧНЫЙ АНАЛИЗ ДАННЫХ

# Замените путь, если ваш CSV-файл лежит в другом месте
data_path = '../data/WA_Fn-UseC_-Telco-Customer-Churn.csv'

df = pd.read_csv(data_path)

print(f"Размер датасета: {df.shape}")
print(df.head())

# %% [3] ОЧИСТКА И ПРЕДОБРАБОТКА ДАННЫХ (DATA CLEANING)

# Удаляем уникальный ID клиента
df = df.drop(columns=['customerID'])

# В колонке TotalCharges есть пробелы ' ' — заменяем их на NaN и приводим к float
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].str.strip(), errors='coerce')

# Заполняем пропуски медианным значением
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# Приводим целевую переменную Churn ('Yes'/'No') в (1/0)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print("\nПропуски после обработки:")
print(df.isnull().sum().sum())

# %% [4] РАЗВЕДКА ДАННЫХ (EDA - ГРАФИКИ)

plt.figure(figsize=(12, 5))

# Распределение оттока
plt.subplot(1, 2, 1)
sns.countplot(data=df, x='Churn', palette='coolwarm')
plt.title('Распределение Churn (0 = Stay, 1 = Churn)')
plt.xlabel('Churn')
plt.ylabel('Количество')

# Связь чека с оттоком
plt.subplot(1, 2, 2)
sns.kdeplot(data=df, x='MonthlyCharges', hue='Churn', common_norm=False, palette='coolwarm', fill=True)
plt.title('Распределение MonthlyCharges по классам Churn')
plt.xlabel('Ежемесячный платеж')

plt.tight_layout()
plt.show()

# Корреляционная матрица
plt.figure(figsize=(6, 4))
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']
sns.heatmap(df[num_cols].corr(), annot=True, cmap='Blues', fmt='.2f')
plt.title('Корреляционная матрица числовых признаков')
plt.show()

# %% [5] РАЗДЕЛЕНИЕ ПРИЗНАКОВ И PREPROCESSING PIPELINE

X = df.drop(columns=['Churn'])
y = df['Churn']

num_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_features = X.select_dtypes(include=['object']).columns.tolist()

# Пайплайн предобработки
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_features)
    ]
)

# Stratified Split (80% train / 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nОбучающая выборка: {X_train.shape}, Тестовая выборка: {X_test.shape}\n")

# %% [6] ОБУЧЕНИЕ БАЗОВЫХ МОДЕЛЕЙ (BASELINE)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42)
}

for name, model in models.items():
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', model)])
    pipeline.fit(X_train, y_train)
    
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    
    auc = roc_auc_score(y_test, y_proba)
    print(f"=== {name} ===")
    print(f"ROC-AUC Score: {auc:.4f}")
    print(classification_report(y_test, y_pred))
    print("-" * 50)

# %% [7] ПОДБОР ГИПЕРПАРАМЕТРОВ (GRIDSEARCHCV)

rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [5, 10, 15],
    'classifier__min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
y_best_proba = best_model.predict_proba(X_test)[:, 1]
best_auc = roc_auc_score(y_test, y_best_proba)

print(f"\nЛучшие параметры Random Forest: {grid_search.best_params_}")
print(f"Улучшенный ROC-AUC на тесте: {best_auc:.4f}")

# %% [8] ПОСТРОЕНИЕ ROC-КРИВОЙ

fpr, tpr, _ = roc_curve(y_test, y_best_proba)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f'Random Forest (Tuned) AUC = {best_auc:.3f}', color='blue')
plt.plot([0, 1], [0, 1], 'k--', label='Random Chance')
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('ROC Curve for Telco Churn Classification')
plt.legend()
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '../data/WA_Fn-UseC_-Telco-Customer-Churn.csv'